# Mapping migration


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

Unable to restore variable 'gbif_ecoregion_gdf', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'elpaso_Temp_F', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'elpaso_df', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'plot_yearly_avg', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>


You will also need to import any libraries you are using in this notebook. In this notebook, you will need some additional packages to make interactive maps with vector data and access pre-defined month names.

In [2]:
### import packages
import os 
import pathlib
import geopandas as gpd
import ee 
import pandas as pd 
import earthpy as et
### libraries for dynamic mapping
import cartopy.crs as ccrs
import panel as pn
import hvplot.pandas as hv

### get month names
import calendar

## Step 5: Make an interactive map of GBIF observations

### Step 5A: Create a simplified `GeoDataFrame` for plotting

Plotting larger files can be time consuming. The code below will streamline plotting with `hvplot` by simplifying the geometry, projecting it to a Mercator projection that is compatible with `geoviews`, and cropping off areas in the Arctic:

1. Simplify the ecoregions with `.simplify(.05)`, and save it back to the `geometry` column.
2. Change the Coordinate Reference System (CRS) to Mercator with `.to_crs(ccrs.Mercator())`.
3. Use the plotting code that is already in the cell to check that the plotting runs quickly (less than a minute) and looks the way you want, making sure to change `gdf` to your `GeoDataFrame` name.

In [3]:
### join the gbif observations to the ecoregion polygons
gbif_ecoregion_gdf = (
    #ecoregions_gdf
    gdf
    
    ### match the CRS of the gbif data and the ecoregions
    .to_crs(gbif_gdf.crs)

    ### find ecoregion for each observation
    .sjoin(
        gbif_gdf,
        how = 'inner', 
        predicate = None)

    ### any additional manipulations you want to do 
    ### (renaming columns, etc)
    .rename(columns= {
        'decimalLatitude': 'lat',
        'decimalLongitude': 'lon',
    })
)

### check it out
gbif_ecoregion_gdf

,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,...,kingdom,phylum,class,order,species,occurrenceID,individualCount,occurrenceStatus,lat,lon
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1745012658,2.0,PRESENT,50.518837,-116.044760
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1782800802,1.0,PRESENT,50.524776,-116.032970
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1781997715,1.0,PRESENT,50.524776,-116.032970
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1758799182,1.0,PRESENT,50.621094,-116.099785
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1764405743,1.0,PRESENT,50.519592,-122.492230
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1838044519,1.0,PRESENT,60.404890,-1.117499
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1838968253,1.0,PRESENT,60.404890,-1.117499
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS2949011428,1.0,PRESENT,60.404890,-1.117499
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1838069664,1.0,PRESENT,60.404890,-1.117499


In [ ]:
### simplify the geometry to speed up processing
gbif_ecoregion_gdf['geometry'] = gbif_ecoregion_gdf.geometry.simplify(tolerance=0.5, preserve_topology=True)
### change the CRS to Mercator for mapping

### check that the plot runs in a reasonable amount of time
#gbif_ecoregion_gdf.hvplot(geo = True, crs = ccrs.Mercator())

In [ ]:
# ### simplify the geometry to speed up processing
# occurrence_df['geometry'] = occurrence_df.geometry.simplify(tolerance=0.05, preserve_topology=True)
# ### change the CRS to Mercator for mapping

# ### check that the plot runs in a reasonable amount of time
# occurrence_df.hvplot(geo = True, crs = ccrs.Mercator())

AttributeError: 'DataFrame' object has no attribute 'geometry'

### Step 5B: Map migration over time

1. If applicable, replace any variable names with the names you defined previously.
2. Replace `column_name_used_for_ecoregion_color` and `column_name_used_for_slider` with the column names you wish to use.
3. Customize your plot with your choice of title, tile source, color map, and size.

Your plot will probably still change months very slowly in your Jupyter notebook, because it calculates each month’s plot as needed. Open up the saved HTML file to see faster performance!

In [ ]:
### prep the data, then combine the occurrence data with the ecoregions

### rename columns in ecoregID to match occurrence_df
#gdf = gdf.rename(columns = {'ECO_NAME': 'ecoregion'})

### make ecoreg ID the index for merge with occurrence_df
gdf = gdf.set_index(['ecoregion'])
### join the occurrences with the plotting GeoDataFrame
occurrence_df = gdf.join(occurrence_df)

In [ ]:
occurrence_df.head()

In [ ]:
### activate panel and hvplot extensions you'll need for the interactive plot
pn.extension()
hv.extension("bokeh")

In [ ]:
### plot the data

### get the plot bounds so they don't change with the slider
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month', 
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### plot occurrence by ecoregion and month
migration_plot = (
    occurrence_gdf
    .hvplot(
        c = 'column_name_used_for_shape_color',
        groupby = 'column_name_used_for_slider',

        ### use background tiles
        geo = True, crs = ccrs.Mercator(), tiles = 'OSM',

        ### add a title
        title = "Your Title Here",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### wrap  plot in the Panel layout
interactive_migration_plot = pn.panel(migration_plot)

### check it out
interactive_migration_plot

Due to some annoying issues with permissions for the basemap, we can't export the map you just made as an .html file with the basemap. So we'll make another version, without a basemap, to export. We'll revisit this for your portfolio assignment to make a nice map with a basemap.

In [ ]:
### make a path for your map to live
map_path = os.path.join(data_dir, "map_name.html")

### make a version for export, without the basemap
export_migration_plot = (
    occurrence_gdf
    .hvplot(
        c = 'column_name_used_for_shape_color',
        groupby = 'column_name_used_for_slider',

        ### set CRS
        geo = True, crs = ccrs.Mercator(),

        ### add a title
        title = "Your Title Here",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### export the map as an .html file
export_migration_plot.save(
    map_path,
    embed = True,
    resources = "inline",
)

### Reflect and Respond #2: Interpret your map
What seasonal trends do you see in the GBIF observations? How does your map align with the broader literature on Veery migration?